In [36]:
x1=10
x2=20
y =1

In [37]:
import pandas as pd
df=pd.read_csv("/home/divyansh/.cache/kagglehub/datasets/manishpatil115/farmer-income-prediction-dataset/versions/1/farmer-income-data.zip/lte_train.csv")
df.head()

,FarmerID,State,REGION,SEX,CITY,Zipcode,DISTRICT,VILLAGE,MARITAL_STATUS,Location,...,Rabi Seasons Agro Ecological Sub Zone in 2020,Rabi Seasons Seasonal average groundwater thickness (cm) in 2020,Rabi Seasons Seasonal average groundwater replenishment rate (cm) in 2020,Night light index,Village score based on socio-economic parameters (Non normalised),Village score based on socio-economic parameters (0 to 100),"Village category based on socio-economic parameters (Good, Average, Poor)",Land Holding Index source (Total Agri Area/ no of people),Road density (Km/ SqKm),Target_Variable/Total Income
0,1002818465057450,MADHYA PRADESH,CENTRAL,M,BARELI,464668,RAISEN,Seoni,M,NaN,...,CENTRAL HIGHLANDS (MALWA AND BUNDELKHAND) HOT...,97.24,19.50,0.95,22.380262,33.527178,Poor,0.773129,0.00,1360000
1,1012300674433870,BIHAR,EAST,M,BANDRA,848125,MUZAFFARPUR,Namapur,M,NaN,...,DECCAN PLATEAU (TELANGANA) AND EASTERN GHATS ...,73.96,16.76,0.97,24.630262,37.173626,Poor,0.454140,0.00,807200
2,1013472263587380,MADHYA PRADESH,CENTRAL,M,MALHARGARH,458556,MANDSAUR,Billaud,M,NaN,...,CENTRAL HIGHLANDS ( MALWA ) GUJARAT PLAIN AND...,90.05,22.44,0.95,19.493313,28.848462,Poor,0.657040,0.00,500000
3,1019525480704050,MAHARASHTRA,WEST,M,RENAPUR,413527,LATUR,Renapur,M,NaN,...,DECCAN PLATU HOT SEMI-ARID ECO-REGION,94.64,21.48,0.98,31.836367,48.852156,Average,0.235615,2.49,558000
4,1021915867444260,MADHYA PRADESH,CENTRAL,F,KHURAI,470117,SAGAR,Singhpur,M,NaN,...,CENTRAL HIGHLANDS (MALWA AND BUNDELKHAND) HOT...,95.90,18.93,0.97,21.327371,31.820817,Poor,0.207264,0.00,800000


In [38]:
df.drop(
    columns=[
        'FarmerID',
        'State',
        'REGION',
        'SEX',
        'CITY',
        'Zipcode',
        'DISTRICT',
        'VILLAGE',
        'MARITAL_STATUS',
        'Location',
        'Rabi Seasons Agro Ecological Sub Zone in 2020',
        'K022-Village category based on socio-economic parameters (Good, Average, Poor)'
        'Address type',
        'Ownership',
        'No_of_Active_Loan_In_Bureau',
        'Avg_Disbursement_Amount_Bureau',
        'Non_Agriculture_Income',
        'Total_Land_For_Agriculture',
        'K022-Village category based on Agri parameters (Good, Average, Poor)',
        'K022-Nearest Mandi Name'
    ],
    inplace=True
)


KeyError: "['K022-Village category based on socio-economic parameters (Good, Average, Poor)Address type'] not found in axis"

In [39]:
target_column = "Target_Variable/Total Income"
feature_columns = [
    column
    for column in df.select_dtypes(include="number").columns
    if column != target_column
]

X = df[feature_columns].values
y = df[target_column].values



In [45]:
import numpy as np

class Perceptron:
    def __init__(self, learning_rate=0.01, epochs=100):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None
        
    def _step_function(self, x):
        return np.where(x >= 0, 1, 0)
        
    def fit(self, X, y):
        num_samples, num_features = X.shape
        
        self.weights = np.random.uniform(-0.5, 0.5, num_features)
        self.bias = np.random.uniform(-0.5, 0.5)
        
        for epoch in range(self.epochs):
                linear_output = np.dot(X, self.weights) + self.bias
                y_predicted = self._step_function(linear_output)
                update = self.lr * (y - y_predicted)
                
                self.weights += update * X.T
                self.bias += update
                
    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return self._step_function(linear_output)


In [ ]:
income_median = np.median(y)
y_binary = (y >= income_median).astype(int)

valid_rows = ~np.isnan(X).any(axis=1)
X_clean = X[valid_rows]
y_clean = y_binary[valid_rows]

X_mean = X_clean.mean(axis=0)
X_std = X_clean.std(axis=0)
X_std[X_std == 0] = 1
X_clean = (X_clean - X_mean) / X_std

split_index = int(len(X_clean) * 0.8)
X_train, X_test = X_clean[:split_index], X_clean[split_index:]
y_train, y_test = y_clean[:split_index], y_clean[split_index:]

model = Perceptron(learning_rate=0.01, epochs=100)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = np.mean(predictions == y_test)
print(f"Test accuracy: {accuracy:.2%}")

ValueError: operands could not be broadcast together with shapes (66,) (66,21637) (66,) 